# Лабораторная работа №2 — работа с датасетом из лабораторной работы №1

В этой работе исходный датасет **не копируется** в `lab02`. Мы используем его
непосредственно из соседнего `lab01`.

Это позволяет не создавать вторую копию исходных 2000+ изображений.

## Что выполняется

- создание CSV со списком файлов, датами и ссылками;
- создание требуемого CSV-аннотации;
- копирование с именами `cat_0000.jpg` / `dog_0000.jpg`;
- копирование со случайными номерами `0..10000`;
- функция `get_next_instance()`;
- классы-итераторы;
- финальные проверки результатов.

## 1. Модульная структура

Файл `dataset_tools.py` содержит функции и классы, которые можно импортировать
в другие программы.

Файл `main.py` содержит настройки, функции верхнего уровня и точку входа.

In [4]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_DIR = Path.cwd()
LAB01_DIR = PROJECT_DIR.parent / "lab01"
DATASET_DIR = LAB01_DIR / "dataset"

print("Текущий проект:", PROJECT_DIR)
print("Исходный dataset:", DATASET_DIR)
print("dataset существует:", DATASET_DIR.exists())

Текущий проект: f:\NTSE\lab02
Исходный dataset: f:\NTSE\lab01\dataset
dataset существует: True


## 2. Импорт функций из собственного модуля

In [5]:
from dataset_tools import (
    ClassIterator,
    DatasetIterator,
    build_annotation_table,
    build_source_files_table,
    copy_with_class_names,
    copy_with_random_names,
    get_class_from_path,
    get_dataset_files,
    get_next_instance,
    reset_instance_iterators,
)

print("Собственный модуль успешно импортирован.")

Собственный модуль успешно импортирован.


## 3. Список файлов, дата и ссылка

В `lab01` уже есть `manifest.jsonl`, поэтому для каждого изображения можно
получить ссылку на исходный файл.

Дополнительно считывается дата изменения файла и переводится в ISO 8601:

```python
datetime.fromtimestamp(timestamp).astimezone().isoformat(timespec="seconds")
```

В итоге таблица содержит:

```text
 date | source_url | file_name | absolute_path | relative_path | output_class
```

In [17]:
SOURCE_MANIFEST = LAB01_DIR / "manifest.jsonl"

source_table = build_source_files_table(
    DATASET_DIR,
    PROJECT_DIR,
    SOURCE_MANIFEST,
)

display(source_table.head())
print("Всего файлов:", len(source_table))
display(source_table["output_class"].value_counts())

,date,source_url,file_name,absolute_path,relative_path,output_class
0,2026-10-05T16:05:41+03:00,https://resizer.mail.ru/p/19d379dc-2822-56ad-9...,0000.jpg,F:\NTSE\lab01\dataset\cat\0000.jpg,../lab01/dataset/cat/0000.jpg,cat
1,2026-10-05T16:07:02+03:00,https://i.ytimg.com/vi/vQtMMlDo_rU/maxresdefau...,0001.jpg,F:\NTSE\lab01\dataset\cat\0001.jpg,../lab01/dataset/cat/0001.jpg,cat
2,2026-10-05T16:07:04+03:00,https://s0.rbk.ru/v6_top_pics/media/img/9/28/3...,0002.jpg,F:\NTSE\lab01\dataset\cat\0002.jpg,../lab01/dataset/cat/0002.jpg,cat
3,2026-10-05T16:07:05+03:00,https://i.ytimg.com/vi/S1hGi5087C8/maxresdefau...,0003.jpg,F:\NTSE\lab01\dataset\cat\0003.jpg,../lab01/dataset/cat/0003.jpg,cat
4,2026-10-05T16:07:06+03:00,https://i.ytimg.com/vi/0KIK8LkkOXk/maxresdefau...,0004.jpg,F:\NTSE\lab01\dataset\cat\0004.jpg,../lab01/dataset/cat/0004.jpg,cat


Всего файлов: 2307


output_class
cat    1200
dog    1107
Name: count, dtype: int64

In [7]:
source_csv = PROJECT_DIR / "annotations" / "source_files.csv"
source_csv.parent.mkdir(parents=True, exist_ok=True)

source_table.to_csv(
    source_csv,
    index=False,
    encoding="utf-8-sig",
)

print("CSV шага 0 сохранён:", source_csv)

CSV шага 0 сохранён: f:\NTSE\lab02\annotations\source_files.csv


## 4. CSV-аннотация

Требование пункта:

1. абсолютный путь;
2. относительный путь относительно Python-проекта;
3. текстовая метка класса.

Получаем таблицу именно с этими тремя колонками.

Так как исходный dataset находится в `lab01`, относительный путь относительно
проекта `lab02` будет иметь вид:

```text
../lab01/dataset/cat/0000.jpg
```

In [8]:
files = get_dataset_files(DATASET_DIR)

annotation_table = build_annotation_table(
    files,
    PROJECT_DIR,
    lambda path: get_class_from_path(path, DATASET_DIR),
)

display(annotation_table.head())

annotation_csv = PROJECT_DIR / "annotations" / "annotations.csv"
annotation_table.to_csv(
    annotation_csv,
    index=False,
    encoding="utf-8-sig",
)

print("Аннотация сохранена:", annotation_csv)

,absolute_path,relative_path,output_class
0,F:\NTSE\lab01\dataset\cat\0000.jpg,../lab01/dataset/cat/0000.jpg,cat
1,F:\NTSE\lab01\dataset\cat\0001.jpg,../lab01/dataset/cat/0001.jpg,cat
2,F:\NTSE\lab01\dataset\cat\0002.jpg,../lab01/dataset/cat/0002.jpg,cat
3,F:\NTSE\lab01\dataset\cat\0003.jpg,../lab01/dataset/cat/0003.jpg,cat
4,F:\NTSE\lab01\dataset\cat\0004.jpg,../lab01/dataset/cat/0004.jpg,cat


Аннотация сохранена: f:\NTSE\lab02\annotations\annotations.csv


## 5. `class_номер.jpg`

Исходный файл:

```text
dataset/cat/0000.jpg
```

становится:

```text
output/class_named_dataset/cat_0000.jpg
```

Здесь класс уже виден в имени файла, но отдельная аннотация всё равно
создаётся, потому что она требуется заданием и сохраняет универсальный
способ определить принадлежность экземпляра.

In [9]:
class_named_dir = PROJECT_DIR / "output" / "class_named_dataset"
class_named_csv = (
    PROJECT_DIR / "annotations" / "class_named_annotations.csv"
)

class_named_table = copy_with_class_names(
    DATASET_DIR,
    class_named_dir,
    PROJECT_DIR,
    class_named_csv,
)

display(class_named_table.head())
print("Скопировано файлов:", len(class_named_table))

,absolute_path,relative_path,output_class
0,F:\NTSE\lab02\output\class_named_dataset\cat_0...,output/class_named_dataset/cat_0000.jpg,cat
1,F:\NTSE\lab02\output\class_named_dataset\cat_0...,output/class_named_dataset/cat_0001.jpg,cat
2,F:\NTSE\lab02\output\class_named_dataset\cat_0...,output/class_named_dataset/cat_0002.jpg,cat
3,F:\NTSE\lab02\output\class_named_dataset\cat_0...,output/class_named_dataset/cat_0003.jpg,cat
4,F:\NTSE\lab02\output\class_named_dataset\cat_0...,output/class_named_dataset/cat_0004.jpg,cat


Скопировано файлов: 2307


In [10]:
print("Примеры имён:")
for path in sorted(class_named_dir.glob("*.jpg"))[:10]:
    print(path.name)

Примеры имён:
cat_0000.jpg
cat_0001.jpg
cat_0002.jpg
cat_0003.jpg
cat_0004.jpg
cat_0005.jpg
cat_0006.jpg
cat_0007.jpg
cat_0008.jpg
cat_0009.jpg


## 6. Случайные номера 0..10000

Теперь имена файлов больше не содержат класс. Поэтому единственным источником информации о классе становится CSV-аннотация.

Для выбора номеров используется `random.sample()`. Она сразу выбирает
неповторяющиеся значения.

In [11]:
random_dir = PROJECT_DIR / "output" / "random_dataset"
random_csv = PROJECT_DIR / "annotations" / "random_annotations.csv"

random_table = copy_with_random_names(
    DATASET_DIR,
    random_dir,
    PROJECT_DIR,
    random_csv,
)

display(random_table.head())

numbers = [int(path.stem) for path in random_dir.glob("*.jpg")]

print("Количество файлов:", len(numbers))
print("Номера уникальны:", len(numbers) == len(set(numbers)))
print(
    "Все номера в диапазоне 0..10000:",
    all(0 <= number <= 10000 for number in numbers),
)

,absolute_path,relative_path,output_class
0,F:\NTSE\lab02\output\random_dataset\3396.jpg,output/random_dataset/3396.jpg,cat
1,F:\NTSE\lab02\output\random_dataset\5533.jpg,output/random_dataset/5533.jpg,cat
2,F:\NTSE\lab02\output\random_dataset\7866.jpg,output/random_dataset/7866.jpg,cat
3,F:\NTSE\lab02\output\random_dataset\409.jpg,output/random_dataset/409.jpg,cat
4,F:\NTSE\lab02\output\random_dataset\541.jpg,output/random_dataset/541.jpg,cat


Количество файлов: 2307
Номера уникальны: True
Все номера в диапазоне 0..10000: True


## 7. `get_next_instance()`

Требование: передаём метку класса и получаем следующий экземпляр этого
класса. Повторов нет, а после окончания возвращается `None`.

Состояние последовательности хранится в модуле, поэтому несколько вызовов
подряд продолжают одну и ту же последовательность.

In [12]:
reset_instance_iterators()

for _ in range(5):
    print(get_next_instance("cat", DATASET_DIR))

f:\NTSE\lab01\dataset\cat\0000.jpg
f:\NTSE\lab01\dataset\cat\0001.jpg
f:\NTSE\lab01\dataset\cat\0002.jpg
f:\NTSE\lab01\dataset\cat\0003.jpg
f:\NTSE\lab01\dataset\cat\0004.jpg


Проверим, что повторов нет, взяв первые 20 экземпляров.

In [13]:
reset_instance_iterators()

received = [
    get_next_instance("cat", DATASET_DIR)
    for _ in range(20)
]
received = [path for path in received if path is not None]

print("Получено:", len(received))
print("Уникальных:", len(set(received)))
print("Повторов:", len(received) - len(set(received)))

Получено: 20
Уникальных: 20
Повторов: 0


## 8. Классы-итераторы

`ClassIterator` позволяет проходить по одному классу циклом `for`.

`DatasetIterator` проходит по всему датасету и возвращает пару:

```text
(метка_класса, путь_к_файлу)
```

In [14]:
cat_iterator = ClassIterator("cat", DATASET_DIR)

print("Первые пять файлов класса cat:")
for index, path in zip(range(5), cat_iterator):
    print(index, path)

Первые пять файлов класса cat:
0 f:\NTSE\lab01\dataset\cat\0000.jpg
1 f:\NTSE\lab01\dataset\cat\0001.jpg
2 f:\NTSE\lab01\dataset\cat\0002.jpg
3 f:\NTSE\lab01\dataset\cat\0003.jpg
4 f:\NTSE\lab01\dataset\cat\0004.jpg


In [15]:
print("Первые десять файлов всего датасета:")

for index, (class_name, path) in zip(
    range(10),
    DatasetIterator(DATASET_DIR),
):
    print(index, class_name, path)

Первые десять файлов всего датасета:
0 cat f:\NTSE\lab01\dataset\cat\0000.jpg
1 cat f:\NTSE\lab01\dataset\cat\0001.jpg
2 cat f:\NTSE\lab01\dataset\cat\0002.jpg
3 cat f:\NTSE\lab01\dataset\cat\0003.jpg
4 cat f:\NTSE\lab01\dataset\cat\0004.jpg
5 cat f:\NTSE\lab01\dataset\cat\0005.jpg
6 cat f:\NTSE\lab01\dataset\cat\0006.jpg
7 cat f:\NTSE\lab01\dataset\cat\0007.jpg
8 cat f:\NTSE\lab01\dataset\cat\0008.jpg
9 cat f:\NTSE\lab01\dataset\cat\0009.jpg


## 9. Сравнение количества файлов в датасетах

Сравним количество файлов в исходном датасете и двух созданных копиях.

Ожидается, что все три количества совпадут.

In [16]:
source_count = len(get_dataset_files(DATASET_DIR))
class_named_count = len(
    list(class_named_dir.glob("*.jpg"))
)
random_count = len(
    list(random_dir.glob("*.jpg"))
)

report = pd.DataFrame(
    [
        ["Исходный dataset", source_count],
        ["class_named_dataset", class_named_count],
        ["random_dataset", random_count],
    ],
    columns=["набор", "количество файлов"],
)

display(report)
print(
    "Копирование выполнено полностью:",
    source_count == class_named_count == random_count,
)

,набор,количество файлов
0,Исходный dataset,2307
1,class_named_dataset,2307
2,random_dataset,2307


Копирование выполнено полностью: True
